In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from math import sqrt

# 1. LOAD DATA

df = pd.read_csv('/content/drive/MyDrive/Dav Datasets/Diseertaion dataset/Brent Oil Futures Historical Data (1).csv')

df = df[['Date', 'Price']]
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# Ensure numeric
df['Price'] = df['Price'].astype(float)

# 2. FEATURE ENGINEERING (IMPORTANT IMPROVEMENT)
df['lag1'] = df['Price'].shift(1)
df['lag7'] = df['Price'].shift(7)
df['rolling_mean_7'] = df['Price'].rolling(7).mean()
df['rolling_std_7'] = df['Price'].rolling(7).std()
df.dropna(inplace=True)

# 3. CROSS VALIDATION SETUP
tscv = TimeSeriesSplit(n_splits=5)

rmse_scores = []
mae_scores = []
r2_scores = []


# 4. MODEL TRAINING LOOP
for train_idx, val_idx in tscv.split(df):

    train = df.iloc[train_idx]
    val = df.iloc[val_idx]

    
    # 5. SCALING
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train[['Price', 'lag1', 'lag7', 'rolling_mean_7', 'rolling_std_7']])
    val_scaled = scaler.transform(val[['Price', 'lag1', 'lag7', 'rolling_mean_7', 'rolling_std_7']])

    # 6. SEQUENCE CREATION
    n_input = 30
    n_features = train_scaled.shape[1]

    train_gen = TimeseriesGenerator(train_scaled, train_scaled[:, 0],
                                    length=n_input, batch_size=32)

    val_gen = TimeseriesGenerator(val_scaled, val_scaled[:, 0],
                                  length=n_input, batch_size=32)

    # 7. IMPROVED LSTM MODEL
    model = Sequential()

    model.add(LSTM(64, return_sequences=True, input_shape=(n_input, n_features)))
    model.add(Dropout(0.2))

    model.add(LSTM(32))
    model.add(Dropout(0.2))

    model.add(Dense(1))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.Huber()
    )

    # 8. EARLY STOPPING (PREVENT OVERFITTING)
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    # 9. TRAIN MODEL
    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=50,
        callbacks=[early_stop],
        verbose=0
    )

    # 10. PREDICTION
    preds = model.predict(val_gen)

    # inverse scaling ONLY target column
    preds_full = np.zeros((len(preds), train_scaled.shape[1]))
    preds_full[:, 0] = preds[:, 0]
    preds_inv = scaler.inverse_transform(preds_full)[:, 0]

    true = val['Price'].values[n_input:]

    # 11. EVALUATION
    rmse = sqrt(mean_squared_error(true, preds_inv))
    mae = mean_absolute_error(true, preds_inv)
    r2 = r2_score(true, preds_inv)

    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)


# 12. FINAL RESULTS
print("Mean RMSE:", np.mean(rmse_scores))
print("Mean MAE:", np.mean(mae_scores))
print("Mean R²:", np.mean(r2_scores))